# Phase 4B — validation-only calibration screening

This notebook screens the five frozen Phase-4A weight/bias pairs using development seed `42` and `nfv3_extended`. Checkpoints are selected by validation average precision and thresholds are selected on validation with `max_f1`. It never loads `test1` or `test2`. Completed candidates are resumable and are not retrained on a normal rerun.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_ROOT = Path('/content/temporalgnn-nids')
REPO_URL = 'https://github.com/tatipar/temporalgnn-nids.git'
BRANCH = 'feat/fair-retrain-clean'

if REPO_ROOT.exists():
    subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'switch', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only', 'origin', BRANCH], check=True)
else:
    subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)],
        check=True,
    )
sys.path.insert(0, str(REPO_ROOT / 'code/python'))

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch-geometric'], check=True)

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
GRAPH_VERSION = 'infiltration_v1_w30_tcpflags_episode_split_v1'
DRIVE_ROOT = Path('/content/drive/MyDrive/nids-fair-retrain')
GRAPH_ROOT = DRIVE_ROOT / 'graphs' / GRAPH_VERSION
CALIBRATION_MANIFEST = DRIVE_ROOT / 'calibration' / GRAPH_VERSION / 'calibration_manifest.json'
RESULTS_ROOT = DRIVE_ROOT / 'phase4b' / GRAPH_VERSION

EPOCHS = 60
HIDDEN_DIM = 64
NODE_DIM = 32
DROPOUT = 0.2
LEARNING_RATE = 1e-3
BATCH_STEPS = 10
PATIENCE = 10
MIN_DELTA = 1e-4
AP_EQUIVALENCE_MARGIN = 0.005
NUM_WORKERS = 2
DEVICE = 'auto'
VERIFY_GRAPH_CHECKSUMS = False

assert (GRAPH_ROOT / 'graph_manifest.json').is_file(), GRAPH_ROOT
assert CALIBRATION_MANIFEST.is_file(), CALIBRATION_MANIFEST
print({
    'graph_root': str(GRAPH_ROOT),
    'calibration_manifest': str(CALIBRATION_MANIFEST),
    'results_root': str(RESULTS_ROOT),
})

## Acceptance tests

Run the complete suite against the exact revision before starting expensive screening.

In [ ]:
test_env = dict(os.environ)
test_env['PYTHONPATH'] = str(REPO_ROOT / 'code/python')
subprocess.run(
    [
        sys.executable, '-m', 'unittest', 'discover',
        '-s', str(REPO_ROOT / 'code/python/tests'), '-p', 'test_*.py', '-v',
    ],
    cwd=REPO_ROOT,
    env=test_env,
    check=True,
)

## Stage 1 — screen every candidate with SimpleMLP

This can take a long time. Rerun the cell after a Colab interruption: candidates with verified completion records and checkpoints are skipped. Do not use `--rerun` during normal recovery.

In [ ]:
base_command = [
    sys.executable,
    str(REPO_ROOT / 'code/python/scripts/screen_calibration.py'),
    '--graph-root', str(GRAPH_ROOT),
    '--calibration-manifest', str(CALIBRATION_MANIFEST),
    '--results-root', str(RESULTS_ROOT),
    '--epochs', str(EPOCHS),
    '--hidden-dim', str(HIDDEN_DIM),
    '--node-dim', str(NODE_DIM),
    '--dropout', str(DROPOUT),
    '--learning-rate', str(LEARNING_RATE),
    '--batch-steps', str(BATCH_STEPS),
    '--patience', str(PATIENCE),
    '--min-delta', str(MIN_DELTA),
    '--ap-equivalence-margin', str(AP_EQUIVALENCE_MARGIN),
    '--num-workers', str(NUM_WORKERS),
    '--device', DEVICE,
]
if VERIFY_GRAPH_CHECKSUMS:
    base_command.append('--verify-checksums')

subprocess.run(base_command + ['--stage', 'mlp', '--plan-only'], cwd=REPO_ROOT, check=True)

In [ ]:
mlp_plan_path = RESULTS_ROOT / 'mlp' / 'screening_plan.json'
mlp_plan = json.loads(mlp_plan_path.read_text(encoding='utf-8'))
print(json.dumps({
    'stage': mlp_plan['stage'],
    'seed': mlp_plan['seed'],
    'feature_profile': mlp_plan['feature_profile'],
    'selection_policy': mlp_plan['selection_policy'],
    'execution': mlp_plan['execution'],
    'candidate_ids': [item['candidate_id'] for item in mlp_plan['configurations']],
}, indent=2))

In [ ]:
# Starts or resumes the five long-running MLP candidates.
subprocess.run(base_command + ['--stage', 'mlp'], cwd=REPO_ROOT, check=True)

In [ ]:
import pandas as pd
from IPython.display import display

mlp_summary_path = RESULTS_ROOT / 'mlp' / 'screening_summary.json'
mlp_summary = json.loads(mlp_summary_path.read_text(encoding='utf-8'))
assert mlp_summary['status'] == 'complete', mlp_summary['missing_candidate_ids']
mlp_table = pd.DataFrame(mlp_summary['ranking'])
display(mlp_table[[
    'candidate_id', 'pos_weight', 'output_bias_init',
    'best_validation_ap', 'best_epoch', 'stopped_epoch',
    'selected_validation_threshold',
]])
print(json.dumps(mlp_summary['ranking_policy'], indent=2))
print('Recommended ST-GNN shortlist:', mlp_summary['recommended_stgnn_shortlist_ids'])
print('Preferred smaller-weight candidate:', mlp_summary['preferred_smaller_weight_candidate_id'])

## Stop and freeze the ST-GNN shortlist

Do not run ST-GNN until the complete MLP summary has been reviewed. The plan predeclares an absolute AP-equivalence margin of `0.005`; every candidate within that distance of the best validation AP enters the recommended shortlist, and the smaller weight is preferred inside the margin. Copy the reviewed shortlist below without consulting any test split. This is still validation-only screening, not a reportable model comparison.

In [ ]:
# Fill only after reviewing and freezing the complete MLP screening summary.
STGNN_CANDIDATE_IDS = []

if not STGNN_CANDIDATE_IDS:
    print('ST-GNN screening is intentionally disabled until the shortlist is frozen.')
else:
    stgnn_command = base_command + ['--stage', 'stgnn']
    for candidate_id in STGNN_CANDIDATE_IDS:
        stgnn_command.extend(['--candidate-id', candidate_id])
    subprocess.run(stgnn_command, cwd=REPO_ROOT, check=True)

In [ ]:
stgnn_summary_path = RESULTS_ROOT / 'stgnn' / 'screening_summary.json'
if stgnn_summary_path.is_file():
    stgnn_summary = json.loads(stgnn_summary_path.read_text(encoding='utf-8'))
    stgnn_table = pd.DataFrame(stgnn_summary['ranking'])
    display(stgnn_table[[
        'candidate_id', 'pos_weight', 'output_bias_init',
        'best_validation_ap', 'best_epoch', 'stopped_epoch',
        'selected_validation_threshold',
    ]])
else:
    print('No ST-GNN summary yet; complete and review the MLP stage first.')